# 特徴量エンジニアリング & データセットクリーン化

運営提供の9種類の合成データセットを収集・分類・サニタイズする。

In [1]:
# =============================================================
# コンプライアンス宣言
# =============================================================
# 本ノートブックにおけるデータ処理は、全てPythonの標準ライブラリ
# および公開パッケージによる機械的な変換・バリデーションのみで行っている。
# LLM（大規模言語モデル）を用いたデータの生成・改変は一切行っていない。
# これはコンペティションのルールを遵守するためである。
# =============================================================

import json
import csv
import io
import re
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

import yaml
from datasets import load_dataset

try:
    import tomllib
except ImportError:
    import tomli as tomllib

print("imports done")

imports done


In [2]:
# =============================================================
# 全データセットの収集 & merge
# =============================================================
#
# 9つのデータセットを統一形式に正規化してmergeする。
# フォーマット判定:
#   - u-10bei系: metadata["format"] をそのまま使用（json, xml 等）
#   - daichira系: category列（C_TOML, C_XML 等）からプレフィクスを除去して使用

DATASETS = [
    # (label, HuggingFace ID, series)
    ("1-1", "u-10bei/structured_data_with_cot_dataset_512_v2",   "u-10bei"),
    ("1-2", "u-10bei/structured_data_with_cot_dataset_512_v4",   "u-10bei"),
    ("1-3", "u-10bei/structured_data_with_cot_dataset_512_v5",   "u-10bei"),
    ("1-4", "u-10bei/structured_data_with_cot_dataset_512",      "u-10bei"),
    ("1-5", "u-10bei/structured_data_with_cot_dataset_v2",       "u-10bei"),
    ("1-6", "u-10bei/structured_data_with_cot_dataset",          "u-10bei"),
    ("2-1", "daichira/structured-3k-mix-sft",                    "daichira"),
    ("2-2", "daichira/structured-5k-mix-sft",                    "daichira"),
    ("2-3", "daichira/structured-hard-sft-4k",                   "daichira"),
]


def normalize_u10bei(example: dict, label: str) -> dict:
    """u-10bei系の正規化。metadata["format"]を出力フォーマットとして使用。"""
    meta = example.get("metadata", {})
    return {
        "source": label,
        "series": "u-10bei",
        "messages": example["messages"],
        "format": meta.get("format", ""),          # 出力フォーマット (json, xml 等)
        "complexity": meta.get("complexity", ""),
        "schema": meta.get("schema", ""),
        "type": meta.get("type", ""),              # conversion, generation 等
    }


def normalize_daichira(example: dict, label: str) -> dict:
    """daichira系の正規化。category列(C_TOML等)を出力フォーマットに変換。"""
    # daichira系はcategory列に C_TOML, C_XML 等の形式でフォーマットが格納されている
    raw_cat = example.get("category", "")
    fmt = raw_cat.replace("C_", "").lower() if raw_cat.startswith("C_") else raw_cat.lower()
    return {
        "source": label,
        "series": "daichira",
        "messages": example["messages"],
        "format": fmt,                              # C_TOML → toml, C_XML → xml
        "complexity": "",
        "schema": example.get("subcategory", ""),   # text_to_toml, json_to_xml 等
        "type": example.get("task", ""),            # extract, transform 等
    }


all_records = []

for label, name, series in DATASETS:
    print(f"Loading [{label}] {name} ...")
    ds = load_dataset(name)
    normalize = normalize_daichira if series == "daichira" else normalize_u10bei
    for split in ds:
        for example in ds[split]:
            all_records.append(normalize(example, label))
    print(f"  -> cumulative: {len(all_records)}")

print(f"\nTotal merged: {len(all_records)} records")

Loading [1-1] u-10bei/structured_data_with_cot_dataset_512_v2 ...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3933 [00:00<?, ? examples/s]

  -> cumulative: 3933
Loading [1-2] u-10bei/structured_data_with_cot_dataset_512_v4 ...


...

  -> cumulative: 31819
Loading [2-3] daichira/structured-hard-sft-4k ...


README.md: 0.00B [00:00, ?B/s]

synthetic_hard_structured_v1.jsonl:   0%|          | 0.00/11.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

  -> cumulative: 35819

Total merged: 35819 records


In [3]:
# =============================================================
# データ概要の確認（フォーマット分布 & トークン数）
# =============================================================
#
# フォーマット判定の優先順位:
#   1. merge時に設定済みのformat列（u-10bei: metadata, daichira: category列由来）
#   2. system promptからの推定（"You are an expert in TOML format" → toml）
#   3. user promptからの推定（"output CSV" 等）
#
# トークン数は学習時のMAX_SEQ_LEN設定の判断材料として算出する。

from transformers import AutoTokenizer

TOKENIZER_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print(f"Tokenizer loaded: {TOKENIZER_NAME}")


def detect_output_format(record: dict) -> str:
    """レコードの出力フォーマットを判定する。
    merge時に設定されたformat列を最優先し、不明な場合のみプロンプトから推定。
    """
    KNOWN_FORMATS = ("csv", "json", "xml", "yaml", "toml")

    # 1. format列（u-10bei: metadata由来, daichira: category列由来）
    fmt = record.get("format", "").lower().strip()
    if fmt in KNOWN_FORMATS:
        return fmt

    # 2. system promptから推定
    for msg in record.get("messages", []):
        if msg.get("role") == "system":
            content = msg["content"].lower()
            for f in KNOWN_FORMATS:
                if f in content:
                    return f

    # 3. user promptから推定（"output TOML" 等のパターン）
    for msg in record.get("messages", []):
        if msg.get("role") == "user":
            content = msg["content"].lower()
            for f in KNOWN_FORMATS:
                if f"output {f}" in content:
                    return f

    return "unknown"


def count_tokens(record: dict) -> int:
    """messagesの全contentを結合しトークン数を返す。"""
    full_text = " ".join(msg["content"] for msg in record.get("messages", []))
    return len(tokenizer.encode(full_text, add_special_tokens=False))


# --- フォーマットを再判定し、トークン数を算出 ---
format_counts = Counter()
token_lengths = []

for rec in all_records:
    fmt = detect_output_format(rec)
    rec["format"] = fmt
    format_counts[fmt] += 1
    n_tokens = count_tokens(rec)
    rec["n_tokens"] = n_tokens
    token_lengths.append(n_tokens)

# --- フォーマット分布 ---
print("=" * 50)
print("出力フォーマット別件数")
print("=" * 50)
for fmt, cnt in format_counts.most_common():
    print(f"  {fmt:10s}: {cnt:>6,}")
print(f"  {'合計':10s}: {len(all_records):>6,}")

# --- トークン数の分布 ---
token_lengths.sort()
n = len(token_lengths)
print(f"\n{'=' * 50}")
print("トークン数の分布")
print("=" * 50)
for label, idx in [("Min", 0), ("25%", n//4), ("50%", n//2),
                    ("75%", 3*n//4), ("90%", int(n*0.9)),
                    ("95%", int(n*0.95)), ("99%", int(n*0.99)), ("Max", -1)]:
    print(f"  {label:>4s}: {token_lengths[idx]}")

# --- 各MAX_SEQ_LENでのカバー率 ---
print(f"\n{'=' * 50}")
print("MAX_SEQ_LEN別カバー率")
print("=" * 50)
for seq_len in [512, 1024, 2048]:
    covered = sum(1 for t in token_lengths if t <= seq_len)
    print(f"  {seq_len:>5}: {covered}/{n} ({covered / n * 100:.1f}%)")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Tokenizer loaded: Qwen/Qwen3-4B-Instruct-2507
出力フォーマット別件数
  xml       :  9,226
  toml      :  7,513
  yaml      :  7,273
  json      :  6,414
  csv       :  5,393
  合計        : 35,819

トークン数の分布
   Min: 98
   25%: 234
   50%: 309
   75%: 593
   90%: 1498
   95%: 1867
   99%: 2209
   Max: 2503

MAX_SEQ_LEN別カバー率
    512: 25349/35819 (70.8%)
   1024: 29663/35819 (82.8%)
   2048: 34898/35819 (97.4%)


In [4]:
# =============================================================
# 出力フォーマット別にレコードを分類
# =============================================================
#
# 命名規則: samples_output_csv = 「CSVとして出力するサンプル群」
#
# 各リスト内のレコードは更に、入力元フォーマット別に集計する。
# 例: samples_output_csv の中に JSON→CSV, TOML→CSV, 自然文→CSV 等が混在する。
#
# 入力元フォーマットの判定:
#   - daichira系: schema列（json_to_csv など）を最優先で機械的に使用
#   - u-10bei系: schemaに変換情報が無い場合のみ userメッセージをフォールバック利用
#
# NOTE:
#   source_format == unknown は「入力元の型を特定できない」ため、
#   怪しいデータとしてこの段階で除外する（厳格運用）。


def detect_source_format(record: dict) -> str:
    """レコードの入力（変換元）フォーマットを推定する。"""

    known_sources = ("csv", "json", "xml", "yaml", "toml", "text")
    series = record.get("series", "")

    # daichira系: schema(subcategory) を最優先で機械的に判定
    # 例: "json_to_xml" -> "json"
    if series == "daichira":
        schema = str(record.get("schema", "")).lower()
        if "_to_" in schema:
            src = schema.split("_to_")[0].strip()
            if src in known_sources:
                return src
        return "unknown"

    # u-10bei系: schemaに "*_to_*" があればそれを使用
    schema = str(record.get("schema", "")).lower()
    if "_to_" in schema:
        src = schema.split("_to_")[0].strip()
        if src in known_sources:
            return src

    # u-10bei系: userメッセージから推定（明示メタが無い場合のフォールバック）
    for msg in record.get("messages", []):
        if msg.get("role") == "user":
            content = msg["content"]
            m = re.match(
                r"(?:Transform|Convert)\s+(?:this|the following)\s+"
                r"(\w+)\s+(?:data\s+)?(?:into|to)\s+\w+",
                content, re.IGNORECASE
            )
            if m:
                src = m.group(1).lower()
                if src in known_sources:
                    return src

            # generation タスク
            if re.match(r"Generate\s+", content, re.IGNORECASE):
                return "generation"

            # extract タスク
            if re.match(r"Extract\s+", content, re.IGNORECASE):
                return "text"
            break

    return "unknown"


# --- 出力フォーマット別リスト ---
samples_output_csv  = []
samples_output_json = []
samples_output_xml  = []
samples_output_yaml = []
samples_output_toml = []
samples_output_unknown = []

FORMAT_SAMPLES = {
    "csv":  samples_output_csv,
    "json": samples_output_json,
    "xml":  samples_output_xml,
    "yaml": samples_output_yaml,
    "toml": samples_output_toml,
}

# --- 分類実行 & 変換パス集計 ---
conversion_paths = Counter()  # 例: "json -> csv": 120
source_unknown_rejected = []
records_known_source = []

for rec in all_records:
    target_fmt = rec["format"]

    # 入力元フォーマットを推定
    src_fmt = detect_source_format(rec)
    rec["source_format"] = src_fmt

    # 厳格運用: source不明は上流で除外
    if src_fmt == "unknown":
        source_unknown_rejected.append((rec, "S1: source_format unknown"))
        continue

    records_known_source.append(rec)

    # 出力フォーマット別リストに追加
    lst = FORMAT_SAMPLES.get(target_fmt)
    if lst is not None:
        lst.append(rec)
    else:
        samples_output_unknown.append(rec)

    # 変換パスを集計
    conversion_paths[f"{src_fmt} -> {target_fmt}"] += 1


# --- レポート: 出力フォーマット別件数 ---
print("=" * 55)
print("出力フォーマット別件数")
print("=" * 55)
for name, lst in FORMAT_SAMPLES.items():
    print(f"  samples_output_{name:4s}: {len(lst):>6,}")
print(f"  samples_output_unknown: {len(samples_output_unknown):>6,}")
print(f"  合計(採用):             {sum(len(v) for v in FORMAT_SAMPLES.values()) + len(samples_output_unknown):>6,}")
print(f"  source_unknown除外:     {len(source_unknown_rejected):>6,}")
print(f"  全体:                   {len(all_records):>6,}")

# --- レポート: 各出力フォーマット内の入力元内訳 ---
print(f"\n{'=' * 55}")
print("出力フォーマット別 入力元の内訳")
print("=" * 55)
for target_fmt, lst in FORMAT_SAMPLES.items():
    src_counter = Counter(rec["source_format"] for rec in lst)
    print(f"\n  [{target_fmt.upper()}] (total: {len(lst)})")
    for src, cnt in src_counter.most_common():
        print(f"    {src:12s}: {cnt:>5,}")

# --- レポート: 変換パス全体 (上位20) ---
print(f"\n{'=' * 55}")
print("変換パス別件数 (上位20)")
print("=" * 55)
for path, count in conversion_paths.most_common(20):
    print(f"  {path:25s}: {count:>5,}")

if source_unknown_rejected:
    unknown_by_series = Counter(rec.get("series", "unknown") for rec, _ in source_unknown_rejected)
    print(f"\n{'=' * 55}")
    print("source_format == unknown の除外内訳")
    print("=" * 55)
    for s, cnt in unknown_by_series.most_common():
        print(f"  {s:10s}: {cnt:>6,}")


出力フォーマット別件数
  samples_output_csv :  2,427
  samples_output_json:  3,115
  samples_output_xml :  5,950
  samples_output_yaml:  4,065
  samples_output_toml:  4,443
  samples_output_unknown:      0
  合計(採用):             20,000
  source_unknown除外:     15,819
  全体:                   35,819

出力フォーマット別 入力元の内訳

  [CSV] (total: 2427)
    generation  :   943
    xml         :   374
    yaml        :   373
    json        :   361
    text        :   275
    toml        :   101

  [JSON] (total: 3115)
    generation  :   976
    xml         :   761
    csv         :   429
    yaml        :   420
    toml        :   309
    text        :   220

  [XML] (total: 5950)
    json        : 2,304
    xml         : 1,600
    generation  :   928
    csv         :   912
    yaml        :   103
    toml        :   103

  [YAML] (total: 4065)
    text        : 1,600
    generation  :   938
    json        :   611
    csv         :   357
    toml        :   351
    xml         :   208

  [TOML] (total: 4443)
  

## CSVデータの定義 (RFC 4180)

以下に、RFC 4180に基づくCSVの原理原則と定義を記す。
サニタイズメソッドはこの定義に基づいて実装する。

### 1. レコードの分離（行の定義）

- **原則**: 各レコード（行）は、改行コードであるCRLF（`0x0D 0x0A`）によって区切られる。
- **例外**: ファイルの最終レコードについては、末尾にCRLFが存在しなくてもよい。
- **実用上の許容**: LF単体（`0x0A`）もバリデーションで許容する。

### 2. フィールドの分離（列の定義）

- **原則**: 各フィールドはカンマ（`,`）で区切る。
- **制約**: 全レコードは同一のフィールド数を持つこと。

### 3. ヘッダー行の扱い

- **原則**: 最初の行はヘッダー行（フィールド名の列挙）として扱える（オプション）。
- **制約**: ヘッダー行も通常のレコードと同じフィールド数・形式に従う。

### 4. フィールドのエンクロージャー

- **原則**: フィールドはダブルクォーテーション（`\"`）で囲むことができる。
- **必須条件**: フィールド値にカンマ、改行、ダブルクォーテーションを含む場合は必ず囲む。

### 5. ダブルクォーテーションのエスケープ

- **原則**: フィールド値内のダブルクォーテーションは `\"\"` で表現する。
- **例**: `He said, \"Hello\"` → `\"He said, \"\"Hello\"\"\"`

### チェック項目

- **C1**: ヘッダー + 1行以上のデータ行が存在すること
- **C2**: 全行のフィールド数がヘッダーと一致すること
- **C3**: ダブルクォートの開閉が対応していること
- **C4**: フィールド途中の裸のダブルクォートがないこと（`\"\"` でエスケープされていること）
- **C5**: カンマ・改行・クォートを含むフィールドが適切にクォートで囲まれていること

In [5]:
# =============================================================
# CSVサニタイザーの実装 (RFC 4180準拠)
# =============================================================
#
# samples_output_csv のassistant出力に対して C1〜C5 のチェックを行う。
# 他フォーマット（JSON, XML, YAML, TOML）のサニタイザーは
# 本セルの結果を確認した上で、後続セルで順次追加する。
#
# 入力元フォーマットについて:
#   samples_output_csv には JSON→CSV, TOML→CSV, 自然文→CSV 等が混在する。
#   入力元が構造化データ（JSON, YAML, TOML等）の場合、Pythonの組み込み
#   パーサー（json.loads等）でdictに変換し、csv.DictWriter でCSV化すれば
#   機械的に正解CSVを生成できる。
#   非LLM的な処理（正規表現、Pythonパーサー等）による機械的変換は
#   コンペティションルール上許容されているため、バリデーション不合格の
#   サンプルに対して機械的再生成を行うことも可能である。
#   本セルではまずバリデーション（検出と除外）を実施し、
#   再生成は必要に応じて後続セルで実装する。


# ----- ユーティリティ -----

def extract_output_block(assistant_content: str) -> str:
    """assistantの応答から 'Output:' 以降のデータ部分を抽出する。
    u-10bei系はCoT(Approach:...Output:)形式、daichira系は直接出力。
    """
    match = re.search(r"Output:\s*\n(.*)", assistant_content, re.DOTALL)
    if match:
        return match.group(1).strip()
    # 'Output:' マーカーがない場合（daichira系等）はそのまま返す
    return assistant_content.strip()


def get_role_content(record: dict, role: str) -> str | None:
    """messagesから指定roleの最初のcontentを取得する。"""
    for msg in record.get("messages", []):
        if msg.get("role") == role:
            return msg["content"]
    return None


def strip_csv_code_fence(text: str) -> str:
    """```csv ... ``` または ``` ... ``` を除去して本体を返す。"""
    s = text.strip()
    m = re.match(r"^```(?:csv)?\s*\n([\s\S]*?)\n```$", s, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return s


# =============================================================
# sanitize_csv: RFC 4180 チェック項目 C1〜C5
# =============================================================

def sanitize_csv(text: str) -> tuple[bool, str]:
    """
    RFC 4180に基づくCSVバリデーション。
    C1〜C5のチェックを順に適用する。
    全てパスすれば (True, "")、違反があれば (False, "C?:理由") を返す。
    """

    # コードフェンスの除去
    text = strip_csv_code_fence(text)

    # -------------------------------------------------------
    # C3: クォートの整合性（最初にチェック。壊れているとパース自体が不能）
    # -------------------------------------------------------
    if text.count('"') % 2 != 0:
        return False, "C3: unclosed double-quote"

    # -------------------------------------------------------
    # パース試行: C1/C2の判定にはパース結果が必要
    # csv.readerはRFC 4180に概ね準拠したパーサー
    # -------------------------------------------------------
    try:
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
    except csv.Error as e:
        return False, f"C3: csv parse error: {e}"

    # -------------------------------------------------------
    # C1: レコード数（ヘッダー + 1行以上のデータ行が必要）
    # -------------------------------------------------------
    if len(rows) < 2:
        return False, f"C1: only {len(rows)} row(s), need header + data"

    # -------------------------------------------------------
    # C2: フィールド数の一貫性（全行がヘッダーと同じフィールド数）
    # -------------------------------------------------------
    header = rows[0]
    header_len = len(header)
    if header_len == 0:
        return False, "C2: empty header row"
    for i, row in enumerate(rows[1:], start=2):
        if len(row) != header_len:
            return False, f"C2: row {i} has {len(row)} fields, expected {header_len}"

    # -------------------------------------------------------
    # C4: ダブルクォーテーションのエスケープ & 位置の正当性
    #
    # csv.readerは寛容にパースするため、生テキストレベルで以下を検査:
    #   - フィールド途中に裸の " がないか（abc"def は不正）
    #   - クォート終了直後にカンマ/行末以外の文字がないか
    #
    # 複数行フィールド（クォート内の改行）に対応するため、
    # 状態変数 in_quoted で現在クォート内かどうかを追跡する。
    # -------------------------------------------------------
    raw_lines = text.replace("\r\n", "\n").replace("\r", "\n").split("\n")
    in_quoted = False

    for line_no, line in enumerate(raw_lines, start=1):
        col = 0
        while col < len(line):
            ch = line[col]

            if in_quoted:
                # クォート内での " の処理
                if ch == '"':
                    # "" （エスケープ）: 次の文字も " ならスキップ
                    if col + 1 < len(line) and line[col + 1] == '"':
                        col += 2
                        continue
                    # クォート終了
                    in_quoted = False
                    col += 1
                    # 閉じクォート直後はカンマ or 行末のみ許容
                    if col < len(line) and line[col] not in (",", "\r"):
                        return False, f"C4: unexpected char after closing quote at line {line_no} col {col + 1}"
                    continue
                col += 1

            else:
                # クォート外での " の処理
                if ch == '"':
                    # クォート開始はフィールド先頭（行頭 or カンマ直後）のみ許容
                    if col == 0 or (col > 0 and line[col - 1] == ","):
                        in_quoted = True
                        col += 1
                        continue
                    # フィールド途中の裸のクォート → 不正
                    return False, f"C4: unquoted double-quote at line {line_no} col {col + 1}"
                col += 1

    # 全行走査後にクォートが閉じていなければ不正
    if in_quoted:
        return False, "C3: unclosed quote at end of data"

    # -------------------------------------------------------
    # C5: 特殊文字を含むフィールドのクォート
    # csv.readerがパースに成功し、C4の検査も通過していれば、
    # カンマ・改行・クォートを含むフィールドは適切に囲まれている。
    # → C4の詳細チェックでカバー済み。
    # -------------------------------------------------------

    return True, ""


# =============================================================
# CSVサニタイズの実行
# =============================================================

csv_clean = []
csv_rejected = []
csv_error_counter = Counter()

for rec in samples_output_csv:
    # assistant応答の取得
    assistant_content = get_role_content(rec, "assistant")
    if not assistant_content:
        csv_rejected.append((rec, "no assistant content"))
        csv_error_counter["no_assistant"] += 1
        continue

    # Output:ブロックの抽出（CoT部分を除去）
    output_text = extract_output_block(assistant_content)
    if not output_text.strip():
        csv_rejected.append((rec, "empty output"))
        csv_error_counter["empty_output"] += 1
        continue

    # RFC 4180 バリデーション
    is_valid, reason = sanitize_csv(output_text)
    if is_valid:
        csv_clean.append(rec)
    else:
        csv_rejected.append((rec, reason))
        # エラー種別（C1〜C5）で集計
        error_key = reason.split(":")[0]
        csv_error_counter[error_key] += 1

# --- レポート ---
total = len(samples_output_csv)
print("=" * 60)
print("CSV SANITIZE REPORT")
print("=" * 60)
print(f"  Total:     {total}")
print(f"  Clean:     {len(csv_clean)}")
print(f"  Rejected:  {len(csv_rejected)}")
if total > 0:
    print(f"  Pass rate: {len(csv_clean) / total * 100:.1f}%")

if csv_error_counter:
    print(f"\n  --- Error breakdown ---")
    for err, cnt in csv_error_counter.most_common():
        print(f"    {err}: {cnt}")

# リジェクトされたサンプルの入力元フォーマット別内訳
if csv_rejected:
    reject_by_src = Counter(rec.get("source_format", "unknown") for rec, _ in csv_rejected)
    print(f"\n  --- Rejected by source format ---")
    for src, cnt in reject_by_src.most_common():
        print(f"    {src}: {cnt}")

# 具体例（最大5件）
if csv_rejected:
    print(f"\n  --- Sample rejected records (up to 5) ---")
    for rec, reason in csv_rejected[:5]:
        src_fmt = rec.get("source_format", "unknown")
        assistant = get_role_content(rec, "assistant") or ""
        output_preview = extract_output_block(assistant)[:150]
        print(f"    [{rec.get('source')}] ({src_fmt}->csv) {reason}")
        print(f"      preview: {output_preview}...")
        print()

CSV SANITIZE REPORT
  Total:     2427
  Clean:     2427
  Rejected:  0
  Pass rate: 100.0%


## JSONデータの定義 (RFC 8259)

以下に、RFC 8259に基づくJSONの原理原則と定義を記す。
サニタイズメソッドはこの定義に基づいて実装する。

### 1. JSONテキストの完全性

- **原則**: 出力は単一のJSONテキスト（オブジェクトまたは配列）として完結していること。
- **実用上の許容**: Markdownコードフェンス（```json ... ```）は除去したうえで判定する。

### 2. 構文妥当性

- **原則**: `json.loads` で構文エラーなくパースできること。
- **制約**: 余分な説明文やコメント等を含めないこと。

### 3. 非有限数値の禁止

- **原則**: RFC 8259に従い `NaN`, `Infinity`, `-Infinity` を禁止する。

### 4. キー重複の禁止

- **原則**: 同一オブジェクト内で同名キーを重複させない。

### 5. トップレベル型

- **原則**: トップレベルは `object` または `array` とする。

### チェック項目

- **J1**: 空出力ではないこと
- **J2**: JSON構文としてパース可能であること
- **J3**: `NaN/Infinity` を含まないこと
- **J4**: オブジェクトキーの重複がないこと
- **J5**: トップレベルが object/array であること


In [6]:
# =============================================================
# JSONサニタイザーの実装 (RFC 8259準拠)
# =============================================================
#
# samples_output_json のassistant出力に対して J1〜J5 のチェックを行う。
# CSVサニタイザーと同じ方針で、まずはバリデーション（検出と除外）を実施する。


def strip_json_code_fence(text: str) -> str:
    """```json ... ``` または ``` ... ``` を除去して本体を返す。"""
    s = text.strip()
    m = re.match(r"^```(?:json)?\s*\n([\s\S]*?)\n```$", s, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return s


def sanitize_json(text: str) -> tuple[bool, str]:
    """
    RFC 8259に基づくJSONバリデーション。
    J1〜J5のチェックを順に適用する。
    全てパスすれば (True, "")、違反があれば (False, "J?:理由") を返す。
    """

    # -------------------------------------------------------
    # J1: 空出力チェック
    # -------------------------------------------------------
    raw = strip_json_code_fence(text)
    if not raw:
        return False, "J1: empty output"

    # -------------------------------------------------------
    # J2/J3/J4: 構文妥当性・非有限数値・重複キー
    # -------------------------------------------------------
    def _reject_non_finite(x: str):
        raise ValueError(f"J3: non-finite number is not allowed: {x}")

    def _no_duplicate_keys(pairs):
        obj = {}
        for k, v in pairs:
            if k in obj:
                raise ValueError(f"J4: duplicate key: {k}")
            obj[k] = v
        return obj

    try:
        parsed = json.loads(
            raw,
            parse_constant=_reject_non_finite,
            object_pairs_hook=_no_duplicate_keys,
        )
    except json.JSONDecodeError as e:
        return False, f"J2: parse error at line {e.lineno} col {e.colno}: {e.msg}"
    except ValueError as e:
        msg = str(e)
        if msg.startswith("J3:") or msg.startswith("J4:"):
            return False, msg
        return False, f"J2: {msg}"

    # -------------------------------------------------------
    # J5: トップレベル型
    # -------------------------------------------------------
    if not isinstance(parsed, (dict, list)):
        return False, f"J5: top-level must be object/array, got {type(parsed).__name__}"

    return True, ""


# =============================================================
# JSONサニタイズの実行
# =============================================================

json_clean = []
json_rejected = []
json_error_counter = Counter()

for rec in samples_output_json:
    # assistant応答の取得
    assistant_content = get_role_content(rec, "assistant")
    if not assistant_content:
        json_rejected.append((rec, "no assistant content"))
        json_error_counter["no_assistant"] += 1
        continue

    # Output:ブロックの抽出（CoT部分を除去）
    output_text = extract_output_block(assistant_content)
    if not output_text.strip():
        json_rejected.append((rec, "empty output"))
        json_error_counter["empty_output"] += 1
        continue

    # RFC 8259 バリデーション
    is_valid, reason = sanitize_json(output_text)
    if is_valid:
        json_clean.append(rec)
    else:
        json_rejected.append((rec, reason))
        error_key = reason.split(":")[0]
        json_error_counter[error_key] += 1

# --- レポート ---
total = len(samples_output_json)
print("=" * 60)
print("JSON SANITIZE REPORT")
print("=" * 60)
print(f"  Total:     {total}")
print(f"  Clean:     {len(json_clean)}")
print(f"  Rejected:  {len(json_rejected)}")
if total > 0:
    print(f"  Pass rate: {len(json_clean) / total * 100:.1f}%")

if json_error_counter:
    print("\n  --- Error breakdown ---")
    for err, cnt in json_error_counter.most_common():
        print(f"    {err}: {cnt}")

# リジェクトされたサンプルの入力元フォーマット別内訳
if json_rejected:
    reject_by_src = Counter(rec.get("source_format", "unknown") for rec, _ in json_rejected)
    print("\n  --- Rejected by source format ---")
    for src, cnt in reject_by_src.most_common():
        print(f"    {src}: {cnt}")

# 具体例（最大5件）
if json_rejected:
    print("\n  --- Sample rejected records (up to 5) ---")
    for rec, reason in json_rejected[:5]:
        src_fmt = rec.get("source_format", "unknown")
        assistant = get_role_content(rec, "assistant") or ""
        output_preview = strip_json_code_fence(extract_output_block(assistant))[:150]
        print(f"    [{rec.get('source')}] ({src_fmt}->json) {reason}")
        print(f"      preview: {output_preview}...")
        print()


JSON SANITIZE REPORT
  Total:     3115
  Clean:     3115
  Rejected:  0
  Pass rate: 100.0%


## YAMLデータの定義 (YAML 1.1 / PyYAML SafeLoader)

以下に、YAML 1.1（PyYAML SafeLoader準拠）に基づくYAMLの原理原則と定義を記す。

> **注**: PyYAMLのSafeLoaderはYAML 1.1仕様に基づいており、`yes`/`no`等がbooleanとして解釈されます。
> YAML 1.2準拠が必要な場合は `ruamel.yaml` 等の使用を検討してください。
サニタイズメソッドはこの定義に基づいて実装する。

### 1. YAMLテキストの完全性

- **原則**: 出力は単一ドキュメントのYAMLとして完結していること。
- **実用上の許容**: Markdownコードフェンス（```yaml / ```yml）を除去したうえで判定する。

### 2. 構文妥当性

- **原則**: `yaml.load(..., Loader=SafeLoader系)` で構文エラーなくパースできること。
- **制約**: 余分な説明文・複数ドキュメント混在は不正とする。

### 3. 空ドキュメントの禁止

- **原則**: `null` 相当の空ドキュメント（`None`）は不正とする。

### 4. キー重複の禁止

- **原則**: 同一マッピング内で同名キーを重複させない。

### 5. トップレベル型

- **原則**: トップレベルは `mapping` または `sequence` とする。

### チェック項目

- **Y1**: 空出力ではないこと
- **Y2**: YAML構文としてパース可能であること
- **Y3**: 空ドキュメント（None）でないこと
- **Y4**: マッピングキーの重複がないこと
- **Y5**: トップレベルが mapping/sequence であること


In [7]:
# =============================================================
# YAMLサニタイザーの実装 (YAML 1.1 / PyYAML SafeLoader準拠)
# =============================================================
#
# samples_output_yaml のassistant出力に対して Y1〜Y5 のチェックを行う。
# CSV/JSONサニタイザーと同じ方針で、まずはバリデーション（検出と除外）を実施する。


def strip_yaml_code_fence(text: str) -> str:
    """```yaml ... ``` / ```yml ... ``` / ``` ... ``` を除去して本体を返す。"""
    s = text.strip()
    m = re.match(r"^```(?:yaml|yml)?\s*\n([\s\S]*?)\n```$", s, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return s


class UniqueKeySafeLoader(yaml.SafeLoader):
    """YAMLロード時に重複キーを検出するためのLoader。"""
    pass


def _construct_unique_mapping(loader, node, deep=False):
    mapping = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise ValueError(f"Y4: duplicate key: {key}")
        mapping[key] = loader.construct_object(value_node, deep=deep)
    return mapping


UniqueKeySafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG,
    _construct_unique_mapping,
)


def sanitize_yaml(text: str) -> tuple[bool, str]:
    """
    YAML 1.2に基づくYAMLバリデーション。
    Y1〜Y5のチェックを順に適用する。
    全てパスすれば (True, "")、違反があれば (False, "Y?:理由") を返す。
    """

    # -------------------------------------------------------
    # Y1: 空出力チェック
    # -------------------------------------------------------
    raw = strip_yaml_code_fence(text)
    if not raw:
        return False, "Y1: empty output"

    # -------------------------------------------------------
    # Y2/Y4: 構文妥当性・重複キー
    # -------------------------------------------------------
    try:
        parsed = yaml.load(raw, Loader=UniqueKeySafeLoader)
    except yaml.YAMLError as e:
        return False, f"Y2: parse error: {e}"
    except ValueError as e:
        msg = str(e)
        if msg.startswith("Y4:"):
            return False, msg
        return False, f"Y2: {msg}"

    # -------------------------------------------------------
    # Y3: 空ドキュメント（None）
    # -------------------------------------------------------
    if parsed is None:
        return False, "Y3: empty/null document"

    # -------------------------------------------------------
    # Y5: トップレベル型
    # -------------------------------------------------------
    if not isinstance(parsed, (dict, list)):
        return False, f"Y5: top-level must be mapping/sequence, got {type(parsed).__name__}"

    return True, ""


# =============================================================
# YAMLサニタイズの実行
# =============================================================

yaml_clean = []
yaml_rejected = []
yaml_error_counter = Counter()

for rec in samples_output_yaml:
    # assistant応答の取得
    assistant_content = get_role_content(rec, "assistant")
    if not assistant_content:
        yaml_rejected.append((rec, "no assistant content"))
        yaml_error_counter["no_assistant"] += 1
        continue

    # Output:ブロックの抽出（CoT部分を除去）
    output_text = extract_output_block(assistant_content)
    if not output_text.strip():
        yaml_rejected.append((rec, "empty output"))
        yaml_error_counter["empty_output"] += 1
        continue

    # YAML 1.2 バリデーション
    is_valid, reason = sanitize_yaml(output_text)
    if is_valid:
        yaml_clean.append(rec)
    else:
        yaml_rejected.append((rec, reason))
        error_key = reason.split(":")[0]
        yaml_error_counter[error_key] += 1

# --- レポート ---
total = len(samples_output_yaml)
print("=" * 60)
print("YAML SANITIZE REPORT")
print("=" * 60)
print(f"  Total:     {total}")
print(f"  Clean:     {len(yaml_clean)}")
print(f"  Rejected:  {len(yaml_rejected)}")
if total > 0:
    print(f"  Pass rate: {len(yaml_clean) / total * 100:.1f}%")

if yaml_error_counter:
    print("\n  --- Error breakdown ---")
    for err, cnt in yaml_error_counter.most_common():
        print(f"    {err}: {cnt}")

# リジェクトされたサンプルの入力元フォーマット別内訳
if yaml_rejected:
    reject_by_src = Counter(rec.get("source_format", "unknown") for rec, _ in yaml_rejected)
    print("\n  --- Rejected by source format ---")
    for src, cnt in reject_by_src.most_common():
        print(f"    {src}: {cnt}")

# 具体例（最大5件）
if yaml_rejected:
    print("\n  --- Sample rejected records (up to 5) ---")
    for rec, reason in yaml_rejected[:5]:
        src_fmt = rec.get("source_format", "unknown")
        assistant = get_role_content(rec, "assistant") or ""
        output_preview = strip_yaml_code_fence(extract_output_block(assistant))[:150]
        print(f"    [{rec.get('source')}] ({src_fmt}->yaml) {reason}")
        print(f"      preview: {output_preview}...")
        print()


YAML SANITIZE REPORT
  Total:     4065
  Clean:     4065
  Rejected:  0
  Pass rate: 100.0%


## XMLデータの定義 (XML 1.0)

以下に、XML 1.0に基づくXMLの原理原則と定義を記す。
サニタイズメソッドはこの定義に基づいて実装する。

### 1. XMLテキストの完全性

- **原則**: 出力は単一ドキュメントのXMLとして完結していること。
- **実用上の許容**: Markdownコードフェンス（```xml ... ```）は除去したうえで判定する。

### 2. 構文妥当性

- **原則**: XMLパーサ（`xml.etree.ElementTree`）で構文エラーなくパースできること。
- **制約**: 余分な説明文や複数ルート要素を含めないこと。

### 3. DTD/ENTITYの禁止

- **原則**: 本前処理では安全性と単純性のため `<!DOCTYPE>` / `<!ENTITY>` を含む入力を不正とする。

### 4. ルート要素の妥当性

- **原則**: ルート要素が存在し、タグ名が空でないこと。

### 5. トップレベル型

- **原則**: トップレベルはXML要素（`Element`）として解釈できること。

### チェック項目

- **X1**: 空出力ではないこと
- **X2**: XML構文としてパース可能であること
- **X3**: `DOCTYPE` / `ENTITY` を含まないこと
- **X4**: ルート要素のタグ名が有効であること
- **X5**: トップレベルがXML要素として取得できること


In [8]:
# =============================================================
# XMLサニタイザーの実装 (XML 1.0準拠)
# =============================================================
#
# samples_output_xml のassistant出力に対して X1〜X5 のチェックを行う。
# CSV/JSON/YAMLサニタイザーと同じ方針で、まずはバリデーション（検出と除外）を実施する。


def strip_xml_code_fence(text: str) -> str:
    """```xml ... ``` または ``` ... ``` を除去して本体を返す。"""
    s = text.strip()
    m = re.match(r"^```(?:xml)?\s*\n([\s\S]*?)\n```$", s, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return s


def sanitize_xml(text: str) -> tuple[bool, str]:
    """
    XML 1.0に基づくXMLバリデーション。
    X1〜X5のチェックを順に適用する。
    全てパスすれば (True, "")、違反があれば (False, "X?:理由") を返す。
    """

    # -------------------------------------------------------
    # X1: 空出力チェック
    # -------------------------------------------------------
    raw = strip_xml_code_fence(text)
    if not raw:
        return False, "X1: empty output"

    # -------------------------------------------------------
    # X3: DTD/ENTITY 禁止
    # -------------------------------------------------------
    upper = raw.upper()
    if "<!DOCTYPE" in upper or "<!ENTITY" in upper:
        return False, "X3: DOCTYPE/ENTITY is not allowed"

    # -------------------------------------------------------
    # X2/X5: 構文妥当性・トップレベル型
    # -------------------------------------------------------
    try:
        root = ET.fromstring(raw)
    except ET.ParseError as e:
        return False, f"X2: parse error: {e}"

    if not isinstance(root, ET.Element):
        return False, "X5: top-level is not an XML element"

    # -------------------------------------------------------
    # X4: ルート要素の妥当性
    # -------------------------------------------------------
    tag = getattr(root, "tag", "")
    if not isinstance(tag, str) or not tag.strip():
        return False, "X4: invalid root tag"

    return True, ""


# =============================================================
# XMLサニタイズの実行
# =============================================================

xml_clean = []
xml_rejected = []
xml_error_counter = Counter()

for rec in samples_output_xml:
    # assistant応答の取得
    assistant_content = get_role_content(rec, "assistant")
    if not assistant_content:
        xml_rejected.append((rec, "no assistant content"))
        xml_error_counter["no_assistant"] += 1
        continue

    # Output:ブロックの抽出（CoT部分を除去）
    output_text = extract_output_block(assistant_content)
    if not output_text.strip():
        xml_rejected.append((rec, "empty output"))
        xml_error_counter["empty_output"] += 1
        continue

    # XML 1.0 バリデーション
    is_valid, reason = sanitize_xml(output_text)
    if is_valid:
        xml_clean.append(rec)
    else:
        xml_rejected.append((rec, reason))
        error_key = reason.split(":")[0]
        xml_error_counter[error_key] += 1

# --- レポート ---
total = len(samples_output_xml)
print("=" * 60)
print("XML SANITIZE REPORT")
print("=" * 60)
print(f"  Total:     {total}")
print(f"  Clean:     {len(xml_clean)}")
print(f"  Rejected:  {len(xml_rejected)}")
if total > 0:
    print(f"  Pass rate: {len(xml_clean) / total * 100:.1f}%")

if xml_error_counter:
    print("\n  --- Error breakdown ---")
    for err, cnt in xml_error_counter.most_common():
        print(f"    {err}: {cnt}")

# リジェクトされたサンプルの入力元フォーマット別内訳
if xml_rejected:
    reject_by_src = Counter(rec.get("source_format", "unknown") for rec, _ in xml_rejected)
    print("\n  --- Rejected by source format ---")
    for src, cnt in reject_by_src.most_common():
        print(f"    {src}: {cnt}")

# 具体例（最大5件）
if xml_rejected:
    print("\n  --- Sample rejected records (up to 5) ---")
    for rec, reason in xml_rejected[:5]:
        src_fmt = rec.get("source_format", "unknown")
        assistant = get_role_content(rec, "assistant") or ""
        output_preview = strip_xml_code_fence(extract_output_block(assistant))[:150]
        print(f"    [{rec.get('source')}] ({src_fmt}->xml) {reason}")
        print(f"      preview: {output_preview}...")
        print()


XML SANITIZE REPORT
  Total:     5950
  Clean:     4280
  Rejected:  1670
  Pass rate: 71.9%

  --- Error breakdown ---
    X2: 1670

  --- Rejected by source format ---
    xml: 1600
    json: 33
    generation: 32
    toml: 3
    csv: 2

  --- Sample rejected records (up to 5) ---
    [1-1] (json->xml) X2: parse error: mismatched tag: line 9, column 2
      preview: <?xml version="1.0" encoding="UTF-8"?>
<api_specification>
<openapi>3.0.0</openapi>
<info>
<title>Organic analyzing strategy API</title>
<version>1.0....

    [1-1] (json->xml) X2: parse error: mismatched tag: line 9, column 2
      preview: <?xml version="1.0" encoding="UTF-8"?>
<api_specification>
<openapi>3.0.0</openapi>
<info>
<title>Horizontal interactive project API</title>
<version>...

    [1-1] (generation->xml) X2: parse error: mismatched tag: line 9, column 2
      preview: <?xml version="1.0" encoding="UTF-8"?>
<api_specification>
<openapi>3.0.0</openapi>
<info>
<title>Re-engineered mobile superstructure API</

## TOMLデータの定義 (TOML 1.0)

以下に、TOML 1.0に基づくTOMLの原理原則と定義を記す。
サニタイズメソッドはこの定義に基づいて実装する。

### 1. TOMLテキストの完全性

- **原則**: 出力は単一のTOMLドキュメントとして完結していること。
- **実用上の許容**: Markdownコードフェンス（```toml ... ```）は除去したうえで判定する。

### 2. 構文妥当性

- **原則**: `tomllib.loads`（または`tomli.loads`）で構文エラーなくパースできること。
- **制約**: 余分な説明文を含めないこと。

### 3. 重複キーの禁止

- **原則**: TOML仕様に従い同一キーの再定義は不可。
- **注記**: この違反はパース時エラーとして検出される。

### 4. 空ドキュメントの禁止

- **原則**: 空文字・空白のみは不正とする。

### 5. トップレベル型

- **原則**: トップレベルは `table`（Python上は `dict`）として解釈できること。

### チェック項目

- **T1**: 空出力ではないこと
- **T2**: TOML構文としてパース可能であること
- **T3**: 重複キー等の仕様違反がないこと（パース時に検出）
- **T4**: 空ドキュメント（空文字）でないこと
- **T5**: トップレベルが table(dict) であること


In [9]:
# =============================================================
# TOMLサニタイザーの実装 (TOML 1.0準拠)
# =============================================================
#
# samples_output_toml のassistant出力に対して T1〜T5 のチェックを行う。
# CSV/JSON/YAML/XMLサニタイザーと同じ方針で、まずはバリデーション（検出と除外）を実施する。


def strip_toml_code_fence(text: str) -> str:
    """```toml ... ``` または ``` ... ``` を除去して本体を返す。"""
    s = text.strip()
    m = re.match(r"^```(?:toml)?\s*\n([\s\S]*?)\n```$", s, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return s


def sanitize_toml(text: str) -> tuple[bool, str]:
    """
    TOML 1.0に基づくTOMLバリデーション。
    T1〜T5のチェックを順に適用する。
    全てパスすれば (True, "")、違反があれば (False, "T?:理由") を返す。
    """

    # -------------------------------------------------------
    # T1/T4: 空出力チェック
    # -------------------------------------------------------
    raw = strip_toml_code_fence(text)
    if not raw:
        return False, "T1: empty output"
    if not raw.strip():
        return False, "T4: empty document"

    # -------------------------------------------------------
    # T2/T3: 構文妥当性・重複キー等
    # -------------------------------------------------------
    try:
        parsed = tomllib.loads(raw)
    except Exception as e:
        msg = str(e)
        return False, f"T2: parse error: {msg}"

    # -------------------------------------------------------
    # T5: トップレベル型
    # -------------------------------------------------------
    if not isinstance(parsed, dict):
        return False, f"T5: top-level must be table(dict), got {type(parsed).__name__}"

    return True, ""


# =============================================================
# TOMLサニタイズの実行
# =============================================================

toml_clean = []
toml_rejected = []
toml_error_counter = Counter()

for rec in samples_output_toml:
    # assistant応答の取得
    assistant_content = get_role_content(rec, "assistant")
    if not assistant_content:
        toml_rejected.append((rec, "no assistant content"))
        toml_error_counter["no_assistant"] += 1
        continue

    # Output:ブロックの抽出（CoT部分を除去）
    output_text = extract_output_block(assistant_content)
    if not output_text.strip():
        toml_rejected.append((rec, "empty output"))
        toml_error_counter["empty_output"] += 1
        continue

    # TOML 1.0 バリデーション
    is_valid, reason = sanitize_toml(output_text)
    if is_valid:
        toml_clean.append(rec)
    else:
        toml_rejected.append((rec, reason))
        error_key = reason.split(":")[0]
        toml_error_counter[error_key] += 1

# --- レポート ---
total = len(samples_output_toml)
print("=" * 60)
print("TOML SANITIZE REPORT")
print("=" * 60)
print(f"  Total:     {total}")
print(f"  Clean:     {len(toml_clean)}")
print(f"  Rejected:  {len(toml_rejected)}")
if total > 0:
    print(f"  Pass rate: {len(toml_clean) / total * 100:.1f}%")

if toml_error_counter:
    print("\n  --- Error breakdown ---")
    for err, cnt in toml_error_counter.most_common():
        print(f"    {err}: {cnt}")

# リジェクトされたサンプルの入力元フォーマット別内訳
if toml_rejected:
    reject_by_src = Counter(rec.get("source_format", "unknown") for rec, _ in toml_rejected)
    print("\n  --- Rejected by source format ---")
    for src, cnt in reject_by_src.most_common():
        print(f"    {src}: {cnt}")

# 具体例（最大5件）
if toml_rejected:
    print("\n  --- Sample rejected records (up to 5) ---")
    for rec, reason in toml_rejected[:5]:
        src_fmt = rec.get("source_format", "unknown")
        assistant = get_role_content(rec, "assistant") or ""
        output_preview = strip_toml_code_fence(extract_output_block(assistant))[:150]
        print(f"    [{rec.get('source')}] ({src_fmt}->toml) {reason}")
        print(f"      preview: {output_preview}...")
        print()


TOML SANITIZE REPORT
  Total:     4443
  Clean:     4443
  Rejected:  0
  Pass rate: 100.0%


## 追加クリーン化① 重複チェック（目的）

### 目的
- 同一/準同一サンプルが複数回含まれることを防ぐ。
- 重複により特定パターンが過学習されるリスクを下げる。

### 実行方法
- `source_format == unknown` を原則除外し、対象集合を絞る。
- 厳密重複を2系統で検出する。
  1) `user + source_format + output` ハッシュ
  2) `output-only` ハッシュ
- さらに SimHash で準重複を検出し、しきい値以内なら除外する。
- 除外は `dedup_rejected` に分離し、通過のみ `dedup_clean` として残す。


In [10]:
# =============================================================
# 追加クリーン化① 重複チェック（実行方法）
# =============================================================

import hashlib

SIMHASH_HAMMING_THRESHOLD = 3
SIMHASH_BANDS = 4
SIMHASH_BAND_BITS = 16  # 4 bands x 16 bits = 64 bits


def _normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def _strip_code_fence(text: str, fmt: str) -> str:
    s = str(text).strip()

    # フォーマット専用フェンスを優先して剥がす
    if fmt == "yaml":
        pat = r"^```(?:yaml|yml)?\s*\n([\s\S]*?)\n```$"
    else:
        pat = rf"^```(?:{re.escape(fmt)})?\s*\n([\s\S]*?)\n```$"

    m = re.match(pat, s, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # 言語指定付きフェンスでも、全体が1ブロックなら剥がす
    m2 = re.match(r"^```[a-zA-Z0-9_-]*\s*\n([\s\S]*?)\n```$", s)
    if m2:
        return m2.group(1).strip()

    return s


def _canonicalize_output_for_hash(output_text: str, fmt: str) -> str:
    body = _strip_code_fence(output_text, fmt)

    if fmt == "json":
        obj = json.loads(body)
        return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

    if fmt == "xml":
        root = ET.fromstring(body)
        return ET.tostring(root, encoding="unicode")

    if fmt == "yaml":
        obj = yaml.safe_load(body)
        return yaml.safe_dump(obj, sort_keys=True, allow_unicode=True)

    if fmt == "toml":
        obj = tomllib.loads(body)
        # TOMLを直接再ダンプする標準関数がないため、同値比較用途としてJSON化
        return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

    if fmt == "csv":
        rows = list(csv.reader(io.StringIO(body)))
        out = io.StringIO()
        writer = csv.writer(out, lineterminator="\n")
        writer.writerows(rows)
        return out.getvalue().strip()

    return _normalize_space(body)


def _simhash64(text: str) -> int:
    """簡易64bit SimHash（非LLM・辞書不要）。"""
    tokens = re.findall(r"[a-z0-9_]+", str(text).lower())
    if not tokens:
        tokens = [str(text).lower()[:128]]

    v = [0] * 64
    for tok in tokens:
        h = int(hashlib.blake2b(tok.encode("utf-8"), digest_size=8).hexdigest(), 16)
        for i in range(64):
            v[i] += 1 if ((h >> i) & 1) else -1

    out = 0
    for i, val in enumerate(v):
        if val >= 0:
            out |= (1 << i)
    return out


def _hamming_distance(a: int, b: int) -> int:
    return (a ^ b).bit_count()


# フォーマット別cleanを統合
quality_pool = []
for fmt, records in {
    "csv": csv_clean,
    "json": json_clean,
    "xml": xml_clean,
    "yaml": yaml_clean,
    "toml": toml_clean,
}.items():
    for rec in records:
        quality_pool.append(rec)

print("=" * 60)
print("DEDUP INPUT SUMMARY")
print("=" * 60)
print(f"  Input records: {len(quality_pool)}")


dedup_clean = []
dedup_rejected = []
dedup_error_counter = Counter()

seen_key_full = {}
seen_key_output = {}
# (band_id, band_value) -> list[(kept_index, simhash64)]
simhash_index = defaultdict(list)

for rec in quality_pool:
    fmt = str(rec.get("format", "")).lower().strip() or "unknown"
    source_fmt = str(rec.get("source_format", "unknown"))

    # 厳格運用: source不明は除外
    if source_fmt == "unknown":
        dedup_rejected.append((rec, "D9: source_format unknown"))
        dedup_error_counter["D9"] += 1
        continue

    user_text = get_role_content(rec, "user") or ""
    assistant_text = get_role_content(rec, "assistant") or ""
    output_text = extract_output_block(assistant_text)

    try:
        canonical_output = _canonicalize_output_for_hash(output_text, fmt)
    except Exception as e:
        dedup_rejected.append((rec, f"D0: canonicalization error: {e}"))
        dedup_error_counter["D0"] += 1
        continue

    # 1) user + source + output の厳密重複
    key_full_material = "\n".join([
        fmt,
        source_fmt,
        _normalize_space(user_text).lower(),
        canonical_output,
    ])
    key_full = hashlib.sha256(key_full_material.encode("utf-8")).hexdigest()
    if key_full in seen_key_full:
        dedup_rejected.append((rec, f"D1: duplicate(full) as kept index {seen_key_full[key_full]}"))
        dedup_error_counter["D1"] += 1
        continue

    # 2) output-only の厳密重複
    key_output_material = "\n".join([fmt, canonical_output])
    key_output = hashlib.sha256(key_output_material.encode("utf-8")).hexdigest()
    if key_output in seen_key_output:
        dedup_rejected.append((rec, f"D2: duplicate(output-only) as kept index {seen_key_output[key_output]}"))
        dedup_error_counter["D2"] += 1
        continue

    # 3) SimHash 準重複
    sim = _simhash64(canonical_output)
    candidate_refs = set()
    for band_id in range(SIMHASH_BANDS):
        band_val = (sim >> (band_id * SIMHASH_BAND_BITS)) & ((1 << SIMHASH_BAND_BITS) - 1)
        for kept_idx, prev_sim in simhash_index[(band_id, band_val)]:
            candidate_refs.add((kept_idx, prev_sim))

    near_dup_kept_idx = None
    for kept_idx, prev_sim in candidate_refs:
        if _hamming_distance(sim, prev_sim) <= SIMHASH_HAMMING_THRESHOLD:
            near_dup_kept_idx = kept_idx
            break

    if near_dup_kept_idx is not None:
        dedup_rejected.append((rec, f"D3: near-duplicate(simhash) as kept index {near_dup_kept_idx}"))
        dedup_error_counter["D3"] += 1
        continue

    # keep
    kept_idx = len(dedup_clean)
    dedup_clean.append(rec)
    seen_key_full[key_full] = kept_idx
    seen_key_output[key_output] = kept_idx

    for band_id in range(SIMHASH_BANDS):
        band_val = (sim >> (band_id * SIMHASH_BAND_BITS)) & ((1 << SIMHASH_BAND_BITS) - 1)
        simhash_index[(band_id, band_val)].append((kept_idx, sim))

print("\n" + "=" * 60)
print("DEDUP REPORT")
print("=" * 60)
print(f"  Input:     {len(quality_pool)}")
print(f"  Kept:      {len(dedup_clean)}")
print(f"  Rejected:  {len(dedup_rejected)}")
if quality_pool:
    print(f"  Keep rate: {len(dedup_clean) / len(quality_pool) * 100:.1f}%")

if dedup_error_counter:
    print("\n  --- Error breakdown ---")
    for k, v in dedup_error_counter.most_common():
        print(f"    {k}: {v}")

if dedup_rejected:
    dup_by_fmt = Counter(rec.get("format", "unknown") for rec, _ in dedup_rejected)
    print("\n  --- Rejected by target format ---")
    for fmt, cnt in dup_by_fmt.most_common():
        print(f"    {fmt}: {cnt}")


DEDUP INPUT SUMMARY
  Input records: 18330

DEDUP REPORT
  Input:     18330
  Kept:      12569
  Rejected:  5761
  Keep rate: 68.6%

  --- Error breakdown ---
    D3: 3950
    D1: 961
    D2: 850

  --- Rejected by target format ---
    toml: 1987
    xml: 1836
    yaml: 1134
    json: 624
    csv: 180


## 追加クリーン化② 余計な文章付加チェック（目的）

### 目的
- 構造化データ以外の説明文（例: `Here is ...`, `Output:`）混入を除外する。
- 出力の前後に余計な文字・文が残るサンプルを厳格に落とす。

### 実行方法
- `dedup_clean` を対象に、コードフェンス境界・先頭/末尾境界・定型説明文を検査する。
- フォーマット別に「開始/終了の境界文字」もチェックする。
- ノイズ混入は `noise_rejected` に分離し、通過データを `noise_clean` として次段に渡す。


In [11]:
# =============================================================
# 追加クリーン化② 余計な文章付加チェック（実行方法）
# =============================================================


def detect_extra_text(output_text: str, fmt: str) -> tuple[bool, str]:
    raw = str(output_text)
    if not raw.strip():
        return False, "N0: empty output"

    # コードフェンスがある場合は、テキスト全体が単一フェンスであることを要求
    if "```" in raw:
        m = re.match(r"^\s*```(?:[a-zA-Z0-9_-]+)?\s*\n([\s\S]*?)\n```\s*$", raw)
        if not m:
            return False, "N1: text outside code fence"
        text = m.group(1)
    else:
        text = raw

    if not text.strip():
        return False, "N0: empty output"

    stripped = text.strip()
    lines = [ln.strip() for ln in stripped.splitlines() if ln.strip()]
    if not lines:
        return False, "N0: empty output"

    first = lines[0]
    last = lines[-1]
    first_l = first.lower()
    last_l = last.lower()

    # 典型説明文を先頭・末尾ともに禁止
    bad_prefix = r"^(here(?: is|'s)|sure|certainly|of course|below is|the following|output|result|answer|final answer|approach|reasoning|explanation|steps?)\b"
    if re.match(bad_prefix, first_l):
        return False, "N2: explanatory prefix"
    if re.match(bad_prefix, last_l):
        return False, "N2: explanatory suffix"

    # フォーマット境界チェック（前後余計文字の厳格検出）
    if fmt == "json":
        if stripped[:1] not in ("{", "[") or stripped[-1:] not in ("}", "]"):
            return False, "N4: json boundary mismatch"

    elif fmt == "xml":
        if not stripped.startswith("<") or not stripped.endswith(">"):
            return False, "N4: xml boundary mismatch"

    elif fmt == "toml":
        # TOMLは明確な閉じ記号が無いため、説明文っぽい末尾を追加検出
        if re.search(r"(?:^|\n)\s*(?:note|comment|explanation)\s*:", stripped.lower()):
            return False, "N5: toml explanatory note"

    elif fmt == "yaml":
        # YAMLは境界が曖昧なので、明示的な説明文ラベルを追加検出
        if re.search(r"(?:^|\n)\s*(?:note|comment|explanation)\s*:", stripped.lower()):
            return False, "N5: yaml explanatory note"

    elif fmt == "csv":
        # CSVはヘッダ1行目がデータ列であることを厳しめに確認
        if re.match(r"^(here|output|result|answer|approach)\b", first_l):
            return False, "N6: csv header looks explanatory"

    return True, ""


noise_clean = []
noise_rejected = []
noise_error_counter = Counter()

for rec in dedup_clean:
    fmt = str(rec.get("format", "")).lower().strip()
    assistant_text = get_role_content(rec, "assistant") or ""
    output_text = extract_output_block(assistant_text)

    ok, reason = detect_extra_text(output_text, fmt)
    if ok:
        noise_clean.append(rec)
    else:
        noise_rejected.append((rec, reason))
        noise_error_counter[reason.split(":")[0]] += 1

print("=" * 60)
print("NOISE CHECK REPORT")
print("=" * 60)
print(f"  Input:     {len(dedup_clean)}")
print(f"  Kept:      {len(noise_clean)}")
print(f"  Rejected:  {len(noise_rejected)}")
if dedup_clean:
    print(f"  Keep rate: {len(noise_clean) / len(dedup_clean) * 100:.1f}%")

if noise_error_counter:
    print("\n  --- Error breakdown ---")
    for k, v in noise_error_counter.most_common():
        print(f"    {k}: {v}")

if noise_rejected:
    by_fmt = Counter(rec.get("format", "unknown") for rec, _ in noise_rejected)
    print("\n  --- Rejected by target format ---")
    for fmt, cnt in by_fmt.most_common():
        print(f"    {fmt}: {cnt}")


NOISE CHECK REPORT
  Input:     12569
  Kept:      12554
  Rejected:  15
  Keep rate: 99.9%

  --- Error breakdown ---
    N5: 14
    N2: 1

  --- Rejected by target format ---
    yaml: 14
    csv: 1


## 追加クリーン化③ 外れ値・壊れデータ整合性チェック（目的）

### 目的
- 文字数・行数・キー数の極端値（p99超）を quarantine/reject し、学習安定性を上げる。
- `messages` 構造破損や制御文字混入、`schema`/`format` 不整合を除去する。

### 実行方法
- `noise_clean` を対象に、出力統計（文字数・行数・キー数）の p99 を算出する。
- p99超は `outlier_rejected` に分離する（厳格運用）。
- 続いて整合性検査を実施し、最終通過を `final_clean_records` に格納する。


In [12]:
# =============================================================
# 追加クリーン化③ 外れ値・壊れデータ整合性チェック（実行方法）
# =============================================================


def _count_struct_keys(obj) -> int:
    if isinstance(obj, dict):
        return len(obj) + sum(_count_struct_keys(v) for v in obj.values())
    if isinstance(obj, list):
        return sum(_count_struct_keys(v) for v in obj)
    return 0


def _count_xml_elements(elem) -> int:
    return 1 + sum(_count_xml_elements(ch) for ch in list(elem))


def _safe_percentile(values: list[int], q: float) -> int:
    if not values:
        return 0
    arr = sorted(values)
    idx = int(round((len(arr) - 1) * q))
    idx = max(0, min(idx, len(arr) - 1))
    return arr[idx]


def estimate_output_metrics(rec: dict) -> tuple[int, int, int]:
    """(char_len, line_count, key_count) を返す。"""
    fmt = str(rec.get("format", "")).lower().strip()
    assistant_text = get_role_content(rec, "assistant") or ""
    output_text = extract_output_block(assistant_text)
    body = _strip_code_fence(output_text, fmt)

    char_len = len(body)
    line_count = max(1, len(body.splitlines()))

    key_count = 0
    try:
        if fmt == "json":
            key_count = _count_struct_keys(json.loads(body))
        elif fmt == "yaml":
            key_count = _count_struct_keys(yaml.safe_load(body))
        elif fmt == "toml":
            key_count = _count_struct_keys(tomllib.loads(body))
        elif fmt == "xml":
            root = ET.fromstring(body)
            key_count = _count_xml_elements(root)
        elif fmt == "csv":
            rows = list(csv.reader(io.StringIO(body)))
            if rows:
                cols = len(rows[0])
                key_count = cols * max(1, len(rows) - 1)
    except Exception:
        key_count = 0

    return char_len, line_count, key_count


# --- outlier thresholds (p99) ---
char_vals = []
line_vals = []
key_vals = []
metrics_cache = {}

for i, rec in enumerate(noise_clean):
    c, l, k = estimate_output_metrics(rec)
    metrics_cache[i] = (c, l, k)
    char_vals.append(c)
    line_vals.append(l)
    key_vals.append(k)

p99_char = _safe_percentile(char_vals, 0.99)
p99_line = _safe_percentile(line_vals, 0.99)
p99_key = _safe_percentile(key_vals, 0.99)

print("=" * 60)
print("OUTLIER THRESHOLDS (p99)")
print("=" * 60)
print(f"  p99_char_len : {p99_char}")
print(f"  p99_line_cnt : {p99_line}")
print(f"  p99_key_cnt  : {p99_key}")


outlier_clean = []
outlier_rejected = []
outlier_error_counter = Counter()

for i, rec in enumerate(noise_clean):
    char_len, line_count, key_count = metrics_cache[i]

    if char_len > p99_char:
        outlier_rejected.append((rec, f"O1: char_len>{p99_char} ({char_len})"))
        outlier_error_counter["O1"] += 1
        continue

    if line_count > p99_line:
        outlier_rejected.append((rec, f"O2: line_count>{p99_line} ({line_count})"))
        outlier_error_counter["O2"] += 1
        continue

    if key_count > p99_key:
        outlier_rejected.append((rec, f"O3: key_count>{p99_key} ({key_count})"))
        outlier_error_counter["O3"] += 1
        continue

    outlier_clean.append(rec)

print("\n" + "=" * 60)
print("OUTLIER FILTER REPORT")
print("=" * 60)
print(f"  Input:     {len(noise_clean)}")
print(f"  Kept:      {len(outlier_clean)}")
print(f"  Rejected:  {len(outlier_rejected)}")
if noise_clean:
    print(f"  Keep rate: {len(outlier_clean) / len(noise_clean) * 100:.1f}%")

if outlier_error_counter:
    print("\n  --- Error breakdown ---")
    for k, v in outlier_error_counter.most_common():
        print(f"    {k}: {v}")


def has_bad_control_chars(text: str) -> bool:
    for ch in str(text):
        o = ord(ch)
        if o < 32 and ch not in ("\n", "\r", "\t"):
            return True
    return False


def check_record_integrity(rec: dict) -> tuple[bool, str]:
    msgs = rec.get("messages")
    if not isinstance(msgs, list) or len(msgs) < 2:
        return False, "B1: invalid messages structure"

    for m in msgs:
        if not isinstance(m, dict):
            return False, "B1: message item is not dict"
        if "role" not in m or "content" not in m:
            return False, "B1: message missing role/content"

    roles = {m.get("role") for m in msgs}
    if "user" not in roles or "assistant" not in roles:
        return False, "B2: missing user/assistant role"

    if msgs[-1].get("role") != "assistant":
        return False, "B3: last message is not assistant"

    assistant_text = get_role_content(rec, "assistant")
    if not isinstance(assistant_text, str) or not assistant_text.strip():
        return False, "B4: empty assistant content"

    output_text = extract_output_block(assistant_text)
    if not output_text.strip():
        return False, "B5: empty extracted output"

    if has_bad_control_chars(output_text):
        return False, "B6: output has control characters"

    # daichira系: schema (src_to_tgt) と format の整合性チェック
    if rec.get("series") == "daichira":
        schema = str(rec.get("schema", "")).lower()
        fmt = str(rec.get("format", "")).lower().strip()
        if "_to_" in schema:
            tgt = schema.split("_to_", 1)[1].strip()
            if tgt in ("csv", "json", "xml", "yaml", "toml") and fmt and fmt != tgt:
                return False, f"B8: schema/format mismatch ({fmt} != {tgt})"

    return True, ""


integrity_clean = []
integrity_rejected = []
integrity_error_counter = Counter()

for rec in outlier_clean:
    ok, reason = check_record_integrity(rec)
    if ok:
        integrity_clean.append(rec)
    else:
        integrity_rejected.append((rec, reason))
        integrity_error_counter[reason.split(":")[0]] += 1

# 最終成果物（strict_modeではなく、これをデフォルト最終出力にする）
final_clean_records = integrity_clean

base_rejected = []
if "source_unknown_rejected" in globals() and isinstance(source_unknown_rejected, list):
    base_rejected.extend(source_unknown_rejected)

final_rejected_records = base_rejected + dedup_rejected + noise_rejected + outlier_rejected + integrity_rejected

print("\n" + "=" * 60)
print("INTEGRITY CHECK REPORT")
print("=" * 60)
print(f"  Input:     {len(outlier_clean)}")
print(f"  Kept:      {len(integrity_clean)}")
print(f"  Rejected:  {len(integrity_rejected)}")
if outlier_clean:
    print(f"  Keep rate: {len(integrity_clean) / len(outlier_clean) * 100:.1f}%")

if integrity_error_counter:
    print("\n  --- Error breakdown ---")
    for k, v in integrity_error_counter.most_common():
        print(f"    {k}: {v}")

print("\n" + "=" * 60)
print("FINAL CLEAN SUMMARY")
print("=" * 60)
print(f"  final_clean_records:    {len(final_clean_records)}")
print(f"  final_rejected_records: {len(final_rejected_records)}")

if final_clean_records:
    final_by_fmt = Counter(rec.get("format", "unknown") for rec in final_clean_records)
    print("\n  --- Final clean by target format ---")
    for fmt, cnt in final_by_fmt.most_common():
        print(f"    {fmt}: {cnt}")


OUTLIER THRESHOLDS (p99)
  p99_char_len : 2996
  p99_line_cnt : 130
  p99_key_cnt  : 270

OUTLIER FILTER REPORT
  Input:     12554
  Kept:      12198
  Rejected:  356
  Keep rate: 97.2%

  --- Error breakdown ---
    O1: 126
    O2: 121
    O3: 109

INTEGRITY CHECK REPORT
  Input:     12198
  Kept:      12198
  Rejected:  0
  Keep rate: 100.0%

FINAL CLEAN SUMMARY
  final_clean_records:    12198
  final_rejected_records: 21951

  --- Final clean by target format ---
    yaml: 2796
    json: 2491
    toml: 2456
    xml: 2318
    csv: 2137


## 追加クリーン化④ CoT除去（Qwen3ネイティブフォーマット対応）

### 目的
- u-10bei系データの `Approach: ... Output:` 形式のCoT（Chain-of-Thought）を除去し、構造化データ本体のみを残す。
- Qwen3-Instruct のネイティブ思考フォーマット（`<think>`）との衝突を回避する。
- daichira系データは元から直接出力のため影響なし。

### 実行方法
- `final_clean_records` のassistant応答に対し、`Output:` マーカー以降のみを抽出する。
- `Output:` マーカーが無い応答（daichira系等）はそのまま維持する。
- 除去後に構文再検証を行い、壊れたものは除外する。

### 学習コード側の変更
- `SFT_MASK_COT = "0"` に設定すること（CoTが無いためマスク不要）。

In [13]:
# =============================================================
# 追加クリーン化④ CoT除去
# =============================================================
# Qwen3-Instruct の <think> フォーマットと、学習データの
# Approach:/Output: フォーマットが衝突するのを防ぐため、
# CoT部分を除去し、構造化データ本体のみを assistant 応答として残す。


def strip_cot_from_messages(messages: list) -> list:
    """messagesリスト内のassistant応答からCoT部分を除去する。
    'Output:' マーカー以降のデータ本体のみを残す。
    マーカーが無い場合（daichira系等）はそのまま返す。
    """
    new_messages = []
    for msg in messages:
        if msg.get("role") == "assistant":
            content = msg.get("content", "")
            match = re.search(r"Output:\s*\n(.*)", content, re.DOTALL)
            if match:
                content = match.group(1).strip()
            new_messages.append({"role": "assistant", "content": content})
        else:
            # user/system メッセージはコピーして保持
            new_messages.append(dict(msg))
    return new_messages


# フォーマット別の再検証関数マップ
_revalidators = {
    "csv":  sanitize_csv,
    "json": sanitize_json,
    "yaml": sanitize_yaml,
    "xml":  sanitize_xml,
    "toml": sanitize_toml,
}


# --- CoT除去の実行 ---
total_input = len(final_clean_records)
cot_stripped_count = 0
cot_empty_after_strip = 0
cot_revalidation_failed = 0

cot_clean = []
cot_rejected = []

for rec in final_clean_records:
    original_messages = rec.get("messages", [])
    new_messages = strip_cot_from_messages(original_messages)

    # 変更があったかチェック
    changed = False
    for orig, new in zip(original_messages, new_messages):
        if orig.get("content") != new.get("content"):
            changed = True
            break
    if changed:
        cot_stripped_count += 1

    # assistant応答が空になっていないか確認
    assistant_content = ""
    for msg in new_messages:
        if msg.get("role") == "assistant":
            assistant_content = msg.get("content", "")
    if not assistant_content.strip():
        cot_empty_after_strip += 1
        cot_rejected.append((rec, "empty after CoT strip"))
        continue

    # 構文再検証: CoT除去後の出力が依然として有効か確認
    fmt = str(rec.get("format", "")).lower().strip()
    validator = _revalidators.get(fmt)
    if validator:
        output_text = extract_output_block(assistant_content)
        is_valid, reason = validator(output_text)
        if not is_valid:
            cot_revalidation_failed += 1
            cot_rejected.append((rec, f"revalidation failed: {reason}"))
            continue

    new_rec = {**rec, "messages": new_messages}
    cot_clean.append(new_rec)

# final_clean_records を更新
final_clean_records = cot_clean
final_rejected_records = final_rejected_records + cot_rejected

# --- レポート ---
print("=" * 60)
print("COT REMOVAL REPORT")
print("=" * 60)
print(f"  Input:                  {total_input}")
print(f"  CoT stripped:           {cot_stripped_count}")
print(f"  Empty after strip:      {cot_empty_after_strip}")
print(f"  Revalidation failed:    {cot_revalidation_failed}")
print(f"  Final clean:            {len(cot_clean)}")
print()
print("NOTE: 学習時に SFT_MASK_COT = '0' を設定してください（CoTマスク不要）")


COT REMOVAL REPORT
  Input:                  12198
  CoT stripped:           4746
  Empty after strip:      0
  Revalidation failed:    0
  Final clean:            12198

NOTE: 学習時に SFT_MASK_COT = '0' を設定してください（CoTマスク不要）


## 追加クリーン化⑤ systemプロンプト除去

### 目的
- 推論コード（標準コード2）ではsystemプロンプトなしで推論されるため、学習データも合わせる。
- 学習時と推論時のメッセージ構造を一致させ、train-inference gapを解消する。

### 実行方法
- `messages` 配列から `role == "system"` のメッセージを全て除去する。
- u-10bei系列（596件程度）が対象。daichira系列は元々systemなし。

In [ ]:
# =============================================================
# 追加クリーン化⑤ systemプロンプト除去
# =============================================================

source_records = cot_clean if 'cot_clean' in dir() and cot_clean else final_clean_records

removed_count = 0
for rec in source_records:
    original_len = len(rec['messages'])
    rec['messages'] = [m for m in rec['messages'] if m['role'] != 'system']
    if len(rec['messages']) < original_len:
        removed_count += 1

# 変数を引き継ぎ
sys_clean = source_records
final_clean_records = sys_clean

print(f'systemプロンプト除去: {removed_count} 件から除去')
print(f'残り: {len(final_clean_records)} 件（全件 user/assistant のみ）')

## 追加クリーン化⑥ 末尾改行の統一

### 目的
- daichira系列のcsv/toml/yamlは末尾に `\n` があり、u-10bei系列は無い。
- 同一フォーマットで末尾改行の有無が混在すると、モデルが矛盾した出力パターンを学習してしまう。
- 全レコードで末尾の不要な空白・改行を統一的に strip する。

### 実行方法
- assistant の出力末尾の `\n` を strip する（JSON/XMLは元々なしなので影響なし）。
- strip 後に空になるレコードがないか安全チェックを行う。

In [ ]:
# =============================================================
# 追加クリーン化⑥ 末尾改行の統一
# =============================================================

source_records = final_clean_records

stripped_count = 0
for rec in source_records:
    for msg in rec['messages']:
        if msg['role'] == 'assistant':
            original = msg['content']
            msg['content'] = original.rstrip('\n')
            if msg['content'] != original:
                stripped_count += 1

# 安全チェック: strip後に空にならないか
empty_after_strip = [
    rec for rec in source_records
    if any(m['role'] == 'assistant' and len(m['content'].strip()) == 0 for m in rec['messages'])
]
if empty_after_strip:
    print(f'WARNING: {len(empty_after_strip)} records have empty assistant after strip!')

final_clean_records = source_records

# 統一確認
from collections import Counter
trailing_check = Counter()
for rec in final_clean_records:
    fmt = rec.get('format', '?')
    for m in rec['messages']:
        if m['role'] == 'assistant':
            has_trailing = m['content'].endswith('\n')
            trailing_check[f'{fmt}/trailing'] += int(has_trailing)
            trailing_check[f'{fmt}/clean'] += int(not has_trailing)

print(f'末尾改行 strip: {stripped_count} 件を修正')
print(f'残り: {len(final_clean_records)} 件')
print(f'\n  統一確認（全て clean であること）:')
for fmt in ['csv', 'json', 'xml', 'yaml', 'toml']:
    t = trailing_check.get(f'{fmt}/trailing', 0)
    c = trailing_check.get(f'{fmt}/clean', 0)
    status = 'OK' if t == 0 else 'NG'
    print(f'    {fmt:6s}: trailing={t}, clean={c}  [{status}]')

## 追加クリーン化⑦ n_tokensの再計算

### 目的
- systemプロンプト除去・末尾改行stripにより、実際のトークン数が `n_tokens` フィールドとずれている。
- 特にu-10bei系列はsystemプロンプト分（約15-20トークン）が含まれたまま。
- 正確な `n_tokens` がないと、MAX_SEQ_LEN超過フィルタ（⑥）が不正確になる。

### 実行方法
- SFTで使用するトークナイザ（Qwen3-4B-Instruct）の `apply_chat_template` を用いて正確に再計算する。
- 再計算後、改めてMAX_SEQ_LEN超過のレコードを除外する。

In [ ]:
# =============================================================
# 追加クリーン化⑦ n_tokensの再計算
# =============================================================

from transformers import AutoTokenizer

BASE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

source_records = final_clean_records

recalc_count = 0
over_limit = []
under_limit = []

for rec in source_records:
    msgs = rec['messages']
    # SFTコードと同じ方法でトークン数を計算
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=False)['input_ids']
    new_n_tokens = len(full_ids)

    old_n_tokens = rec.get('n_tokens', 0) or 0
    if new_n_tokens != old_n_tokens:
        recalc_count += 1
    rec['n_tokens'] = new_n_tokens

    if new_n_tokens > MAX_SEQ_LEN:
        over_limit.append((rec, f'n_tokens={new_n_tokens} > MAX_SEQ_LEN={MAX_SEQ_LEN}'))
    else:
        under_limit.append(rec)

# 超過レコードを除外
final_clean_records = under_limit

if 'final_rejected_records' not in dir():
    final_rejected_records = []
final_rejected_records.extend(over_limit)

from collections import Counter
fmt_removed = Counter(r[0].get('format', '?') for r in over_limit)
fmt_remaining = Counter(r.get('format', '?') for r in final_clean_records)

print(f'n_tokens再計算: {recalc_count} 件で値が変化')
print(f'MAX_SEQ_LEN({MAX_SEQ_LEN})超過: {len(over_limit)} 件を除外')
print(f'残り: {len(final_clean_records)} 件')

if over_limit:
    print(f'\n  除外内訳（フォーマット別）:')
    for fmt, cnt in fmt_removed.most_common():
        print(f'    {fmt:8s}: {cnt} 件')

print(f'\n  残りデータ分布:')
for fmt, cnt in fmt_remaining.most_common():
    print(f'    {fmt:8s}: {cnt} 件')

# トークン数分布
all_tokens = [r.get('n_tokens', 0) for r in final_clean_records]
print(f'\n  トークン数: min={min(all_tokens)}, max={max(all_tokens)}, mean={sum(all_tokens)/len(all_tokens):.0f}')

## 追加クリーン化⑧ typeフィールド補完

### 目的
- u-10bei系列の130件で `type` フィールドが空文字になっている。
- `source_format` が `generation` のレコードは `type` も `generation` であるべき。
- フィルタ・分析・デバッグ時の一貫性を確保する。

### 実行方法
- `type` が空で `source_format == "generation"` のレコードに `type = "generation"` を補完する。

In [ ]:
# =============================================================
# 追加クリーン化⑧ typeフィールド補完
# =============================================================

source_records = final_clean_records

fixed_count = 0
still_empty = 0

for rec in source_records:
    if not rec.get('type', '').strip():
        sf = rec.get('source_format', '').strip()
        if sf == 'generation':
            rec['type'] = 'generation'
            fixed_count += 1
        else:
            still_empty += 1

final_clean_records = source_records

print(f'type補完: {fixed_count} 件を "generation" に設定')
if still_empty > 0:
    print(f'WARNING: まだ {still_empty} 件の type が空です')
print(f'残り: {len(final_clean_records)} 件')

## 追加クリーン化⑨ generationタスクの重複プロンプト削減

### 目的
- generationタスクは574件中ユニークプロンプトがわずか108個。
- 同じプロンプトに対して異なる出力を大量に学習すると、モデルが不安定になる。
- 同一プロンプトから最大N件のみ残し、多様性とデータ効率のバランスを取る。

### 実行方法
- generationタスクの同一userプロンプトから最大2件のみ残す。
- 残す際はトークン数が多い（複雑な）サンプルを優先する。

In [ ]:
# =============================================================
# 追加クリーン化⑨ generationタスクの重複プロンプト削減
# =============================================================

from collections import defaultdict

MAX_PER_PROMPT = 2  # 同一プロンプトから残す最大件数

source_records = final_clean_records

gen_records = [r for r in source_records if r.get('type') == 'generation']
non_gen_records = [r for r in source_records if r.get('type') != 'generation']

# プロンプト別にグループ化
prompt_groups = defaultdict(list)
for rec in gen_records:
    user_prompt = next(m['content'] for m in rec['messages'] if m['role'] == 'user')
    prompt_groups[user_prompt].append(rec)

# 各グループからトークン数が多い順にMAX_PER_PROMPT件を選択
kept_gen = []
removed_gen = []
for prompt, recs in prompt_groups.items():
    recs_sorted = sorted(recs, key=lambda r: r.get('n_tokens', 0), reverse=True)
    kept_gen.extend(recs_sorted[:MAX_PER_PROMPT])
    removed_gen.extend(recs_sorted[MAX_PER_PROMPT:])

final_clean_records = non_gen_records + kept_gen

if 'final_rejected_records' not in dir():
    final_rejected_records = []
final_rejected_records.extend([(r, 'duplicate_generation_prompt') for r in removed_gen])

from collections import Counter
fmt_remaining = Counter(r.get('format', '?') for r in final_clean_records)

print(f'generation重複削減: {len(gen_records)} → {len(kept_gen)} 件（{len(removed_gen)} 件除外）')
print(f'ユニークプロンプト: {len(prompt_groups)} 個')
print(f'残り: {len(final_clean_records)} 件')
print(f'\n  フォーマット別残り:')
for fmt, cnt in fmt_remaining.most_common():
    print(f'    {fmt:8s}: {cnt} 件')

## 追加クリーン化⑩ 低品質スキーマ・極短レコード除外

### 目的
- `random_structured_data` スキーマは意味のないランダム文字列で、実際のベンチマークには出ないパターン。
- 極端に短い出力（assistant < 50文字）は学習への寄与が低く、ノイズになりうる。

### 実行方法
- `schema == "random_structured_data"` のレコードを除外する。
- assistant出力が50文字未満のレコードを除外する。

In [ ]:
# =============================================================
# 追加クリーン化⑩ 低品質スキーマ・極短レコード除外
# =============================================================

MIN_ASSISTANT_CHARS = 50  # assistant出力の最小文字数
EXCLUDE_SCHEMAS = {'random_structured_data'}  # 除外するスキーマ

source_records = final_clean_records

clean = []
rejected_schema = []
rejected_short = []

for rec in source_records:
    schema = rec.get('schema', '')
    if schema in EXCLUDE_SCHEMAS:
        rejected_schema.append((rec, f'excluded_schema={schema}'))
        continue

    asst_content = next((m['content'] for m in rec['messages'] if m['role'] == 'assistant'), '')
    if len(asst_content) < MIN_ASSISTANT_CHARS:
        rejected_short.append((rec, f'assistant_too_short={len(asst_content)}chars'))
        continue

    clean.append(rec)

final_clean_records = clean

if 'final_rejected_records' not in dir():
    final_rejected_records = []
final_rejected_records.extend(rejected_schema)
final_rejected_records.extend(rejected_short)

from collections import Counter
fmt_remaining = Counter(r.get('format', '?') for r in final_clean_records)

print(f'スキーマ除外: {len(rejected_schema)} 件（{EXCLUDE_SCHEMAS}）')
print(f'極短除外: {len(rejected_short)} 件（< {MIN_ASSISTANT_CHARS}文字）')
print(f'残り: {len(final_clean_records)} 件')
print(f'\n  フォーマット別残り:')
for fmt, cnt in fmt_remaining.most_common():
    print(f'    {fmt:8s}: {cnt} 件')

## 追加クリーン化⑪ extract/conversionの同一入力重複チェック

### 目的
- extract/conversionでも完全に同一のuserプロンプトがある場合、冗長。
- 同一入力に対しては1件のみ残す（出力は一意に決まるべきタスクのため）。

### 実行方法
- extract/conversionタスクで同一userプロンプトがあれば1件のみ残す。

In [ ]:
# =============================================================
# 追加クリーン化⑪ extract/conversionの同一入力重複チェック
# =============================================================

source_records = final_clean_records

seen_prompts = {}  # (type, user_prompt) -> record
clean = []
dedup_removed = []

for rec in source_records:
    typ = rec.get('type', '')
    if typ in ('extract', 'conversion'):
        user_prompt = next(m['content'] for m in rec['messages'] if m['role'] == 'user')
        key = (typ, user_prompt)
        if key not in seen_prompts:
            seen_prompts[key] = rec
            clean.append(rec)
        else:
            dedup_removed.append((rec, 'duplicate_extract_conversion_prompt'))
    else:
        clean.append(rec)

final_clean_records = clean

if 'final_rejected_records' not in dir():
    final_rejected_records = []
final_rejected_records.extend(dedup_removed)

print(f'extract/conversion重複除外: {len(dedup_removed)} 件')
print(f'残り: {len(final_clean_records)} 件')

## 追加クリーン化⑫ フォーマット均等サンプリング

### 目的
- フォーマット間のデータ量の偏りを解消し、学習バランスを改善する。
- 特にTOML等の少数フォーマットが埋もれるのを防ぐ。
- データ量を1500件程度（300件×5フォーマット）に厳選し、過学習・破滅的忘却を抑制する。

### 実行方法
- 各フォーマット（CSV, JSON, XML, YAML, TOML）から均等に `SAMPLES_PER_FORMAT` 件をランダム抽出する。
- 指定件数に満たないフォーマットは全件使用する。
- 抽出後のデータを `final_clean_records` として次段に渡す。

### 設定
- `SAMPLES_PER_FORMAT`: フォーマットあたりのサンプル数（デフォルト: 300）
- 変更したい場合は下のセルの値を書き換える。

In [ ]:
# =============================================================
# 追加クリーン化⑫ フォーマット均等サンプリング
# =============================================================

import random

SAMPLES_PER_FORMAT = 300  # フォーマットあたりのサンプル数
SAMPLING_SEED = 3407

random.seed(SAMPLING_SEED)

source_records = final_clean_records

TARGET_FORMATS = ["csv", "json", "xml", "yaml", "toml"]

# --- フォーマット別に分類 ---
format_groups = {fmt: [] for fmt in TARGET_FORMATS}
other_records = []

for rec in source_records:
    fmt = rec.get("format", "").lower().strip()
    if fmt in format_groups:
        format_groups[fmt].append(rec)
    else:
        other_records.append(rec)

# --- 均等サンプリング ---
balanced_records = []

print("=" * 50)
print(f"フォーマット均等サンプリング（目標: {SAMPLES_PER_FORMAT}件/フォーマット）")
print("=" * 50)

for fmt in TARGET_FORMATS:
    available = format_groups[fmt]
    if len(available) >= SAMPLES_PER_FORMAT:
        sampled = random.sample(available, SAMPLES_PER_FORMAT)
        balanced_records.extend(sampled)
        print(f"  {fmt:6s}: {len(available):>5} 件中 {SAMPLES_PER_FORMAT} 件を抽出")
    else:
        balanced_records.extend(available)
        print(f"  {fmt:6s}: {len(available):>5} 件 → 全件使用（目標未達）")

# シャッフルして順序バイアスを除去
random.shuffle(balanced_records)

# final_clean_records を上書き
final_clean_records = balanced_records

print(f"\n  合計: {len(final_clean_records)} 件")
if other_records:
    print(f"  除外（unknown等）: {len(other_records)} 件")

## 最終出力 JSONL書き出し（目的）

### 目的
- 追加クリーン化後の全レコードを、学習で再利用しやすい単一JSONLとして保存する。
- 併せて、除外レコードを理由付きで別JSONLに保存し、後で監査・再判定できるようにする。

### 実行方法
- `final_clean_records` を優先し、未定義なら直前段のクリーン変数へフォールバックする。
- 出力ファイル: `merged_dataset_final_clean.jsonl`
- 参考ファイル（除外データ）: `merged_dataset_final_rejected.jsonl`


In [14]:
# =============================================================
# 最終出力 JSONL書き出し（実行方法）
# =============================================================

from pathlib import Path

OUT_FINAL_CLEAN_JSONL = Path("merged_dataset_final_clean.jsonl")
OUT_FINAL_REJECT_JSONL = Path("merged_dataset_final_rejected.jsonl")


def _pick_final_clean_records():
    """Notebook内で利用可能な最新のクリーン集合を選ぶ。"""
    for name in [
        "final_clean_records",
        "integrity_clean",
        "noise_clean",
        "dedup_clean",
        "quality_pool",
    ]:
        if name in globals() and isinstance(globals()[name], list):
            return name, globals()[name]
    raise RuntimeError("No clean record list found. Run cleaning cells first.")


def _pick_final_rejected_records():
    """Notebook内で利用可能な除外集合を選ぶ。"""
    if "final_rejected_records" in globals() and isinstance(final_rejected_records, list):
        return "final_rejected_records", final_rejected_records

    merged = []
    for name in ["dedup_rejected", "noise_rejected", "integrity_rejected"]:
        if name in globals() and isinstance(globals()[name], list):
            merged.extend(globals()[name])
    return "merged_rejected", merged


def _to_training_record(rec: dict) -> dict:
    """学習用JSONLとして必要な項目を残して整形する。"""
    return {
        "source": rec.get("source", ""),
        "series": rec.get("series", ""),
        "messages": rec.get("messages", []),
        "format": rec.get("format", ""),
        "source_format": rec.get("source_format", "unknown"),
        "complexity": rec.get("complexity", ""),
        "schema": rec.get("schema", ""),
        "type": rec.get("type", ""),
        "n_tokens": rec.get("n_tokens", None),
    }


clean_name, clean_records = _pick_final_clean_records()
reject_name, reject_records = _pick_final_rejected_records()

# --- clean JSONL write ---
with OUT_FINAL_CLEAN_JSONL.open("w", encoding="utf-8") as f:
    for rec in clean_records:
        row = _to_training_record(rec)
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# --- reject JSONL write (reason付き) ---
with OUT_FINAL_REJECT_JSONL.open("w", encoding="utf-8") as f:
    for item in reject_records:
        if isinstance(item, tuple) and len(item) == 2:
            rec, reason = item
            row = {
                "reason": reason,
                "record": _to_training_record(rec) if isinstance(rec, dict) else rec,
            }
        else:
            row = {"reason": "unknown", "record": item}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# --- report ---
fmt_counter = Counter(rec.get("format", "unknown") for rec in clean_records)
series_counter = Counter(rec.get("series", "unknown") for rec in clean_records)

print("=" * 60)
print("FINAL JSONL EXPORT REPORT")
print("=" * 60)
print(f"  Source clean list:   {clean_name}")
print(f"  Source reject list:  {reject_name}")
print(f"  Clean records:       {len(clean_records):,}")
print(f"  Rejected records:    {len(reject_records):,}")
print(f"  Clean JSONL path:    {OUT_FINAL_CLEAN_JSONL.resolve()}")
print(f"  Reject JSONL path:   {OUT_FINAL_REJECT_JSONL.resolve()}")

print("\n  --- Clean by format ---")
for fmt, cnt in fmt_counter.most_common():
    print(f"    {fmt:8s}: {cnt:>7,}")

print("\n  --- Clean by series ---")
for series, cnt in series_counter.most_common():
    print(f"    {series:8s}: {cnt:>7,}")


FINAL JSONL EXPORT REPORT
  Source clean list:   final_clean_records
  Source reject list:  final_rejected_records
  Clean records:       12,198
  Rejected records:    21,951
  Clean JSONL path:    /content/merged_dataset_final_clean.jsonl
  Reject JSONL path:   /content/merged_dataset_final_rejected.jsonl

  --- Clean by format ---
    yaml    :   2,796
    json    :   2,491
    toml    :   2,456
    xml     :   2,318
    csv     :   2,137

  --- Clean by series ---
    daichira:   7,452
    u-10bei :   4,746
